# Doc2Query (2019)
[[paper]](https://arxiv.org/abs/1904.08375)<br>
Doc2Query = Document Expansion by Query Prediction

### Определение
Doc2Query — это метод Document Expansion для систем информационного поиска, который использует нейросетевую генеративную модель для предсказания потенциальных поисковых запросов к документу и добавления их в текст документа перед индексацией. Это позволяет улучшить качество Sparse Retrieval (например, BM25) за счет расширения семантического описания контента.

### Задача
Решается задача первого этапа поиска (First Stage Retrieval). Цель — максимально повысить полноту (Recall) выдачи, чтобы релевантные документы попали в топ-K, который затем будет обрабатываться более тяжелыми моделями (Re-rankers).

### Мотивация
Основная проблема классического текстового поиска — Vocabulary Mismatch. Авторы документов и пользователи, задающие вопросы, часто используют разные слова для описания одних и тех же сущностей или концепций. Поскольку BM25 и TF-IDF работают на точном совпадении токенов, документ может быть не найден, если в нем нет конкретных слов из запроса, хотя по смыслу он идеально подходит.

### Альтернативы
На момент 2019 года существовали следующие подходы:
- BM25 (1990-е): классика на основе частотности слов. Ограничен точным совпадением термов.
- Pseudo-Relevance Feedback (PRF), например RM3: расширение запроса на этапе поиска. Берет топ-10 результатов первого прогона, извлекает из них частые слова и добавляет в запрос. Проблема: увеличивает время ответа (latency) и подвержен Query Drift (если в топ-10 попал мусор, запрос испортится).
- Word2Vec/GloVe Expansion: расширение документа синонимами из статических эмбеддингов. Проблема: отсутствие контекста (слово "замок" может значить и здание, и устройство на двери).
- Dense Retrieval (только зарождался): использование векторных представлений. Требовал сложной инфраструктуры для поиска по векторам (HNSW, FAISS), которая на тот момент была менее зрелой, чем инвертированный индекс.

### Идея
Вместо того чтобы пытаться угадать синонимы или расширять запрос во время поиска, авторы предложили обучить модель предсказывать, на какие вопросы этот документ мог бы дать ответ. Эти предсказанные вопросы (запросы) дописываются в конец документа. Таким образом, документ "насыщается" словами, которые с высокой вероятностью будут в реальных запросах пользователей, что позволяет классическому BM25 находить его по этим словам.

### Архитектура
Doc2Query базируется на архитектуре Seq2Seq (Encoder-Decoder):
1. Encoder: принимает на вход текст документа (или его фрагмент). В оригинальной статье 2019 года использовался стандартный Transformer.
2. Decoder: генерирует последовательность токенов, представляющих собой вероятный поисковый запрос.
3. Позже (в версии DocTTTTTquery) архитектура была заменена на предобученную модель T5 (2019, Google), что значительно улучшило качество генерации за счет мощных языковых знаний.

### Обучение
Модель обучается как классическая задача Machine Translation, где "язык документа" переводится в "язык запроса":
1. Используется датасет MS MARCO, содержащий сотни тысяч пар (Document, Relevant Query).
2. Input: текст документа.
3. Target: реальный поисковый запрос, по которому этот документ был найден пользователем.
4. Loss Function: стандартная Cross-Entropy Loss для генеративных задач.

### Инференс
Процесс применения Doc2Query полностью выполняется Offline (до того, как пользователь придет с запросом):
1. Документ подается в обученную Seq2Seq модель.
2. Используется Top-k Sampling или Beam Search для генерации нескольких (обычно 10-40) различных запросов для одного документа.
3. Сгенерированные запросы конкатенируются с оригинальным текстом документа.
4. Расширенный текст индексируется стандартным поисковым движком (Lucene, Elasticsearch, Solr).
5. При поиске используется обычный BM25. Поскольку в индексе теперь есть предсказанные слова, вероятность мэтчинга семантически близких запросов резко возрастает.

### Результаты
Эксперименты проводились на наборах данных MS MARCO и TREC CAR:
- Применение Doc2Query поверх BM25 увеличило метрику Mean Reciprocal Rank (MRR@10) с 18.7% до 21.5% (улучшение на 2.8 п.п. или ~15% относительно базового уровня).
- Recall@1000 вырос значительно, что позволило последующим стадиям переранжирования (BERT re-rankers) работать с более качественным набором кандидатов.
- Главное преимущество: нулевое влияние на Latency поиска. Весь оверхед по вычислениям перенесен на этап индексации, а сам поиск в Elasticsearch остается таким же быстрым, как и раньше.

## 📝 Критический анализ

```markdown
# Doc2Query (2019)
---
[[paper]](https://arxiv.org/abs/1904.08375)<br>
Doc2Query = Document Expansion by Query Prediction

### Определение
Doc2Query — метод Document Expansion для информационного поиска, использующий нейросетевую модель для предсказания потенциальных запросов к документу и добавления их в текст перед индексацией. Это улучшает качество Sparse Retrieval, например, BM25, расширяя семантическое описание контента.

### Задача
Решается задача First Stage Retrieval, цель — повысить Recall, чтобы релевантные документы попали в топ-K для дальнейшей обработки более сложными моделями.

### Мотивация
Классический поиск страдает от Vocabulary Mismatch: авторы и пользователи используют разные слова для одних и тех же понятий. BM25 и TF-IDF зависят от точного совпадения токенов, что может привести к пропуску релевантных документов.

### Альтернативы
На 2019 год существовали:
- BM25: ограничен точным совпадением термов.
- Pseudo-Relevance Feedback (PRF): расширяет запрос, но увеличивает latency и подвержен Query Drift.
- Word2Vec/GloVe Expansion: расширяет документ синонимами, но без учета контекста.
- Dense Retrieval: требовал сложной инфраструктуры для поиска по векторам.

### Идея
Авторы предложили обучить модель предсказывать вопросы, на которые документ мог бы ответить. Эти вопросы добавляются в конец документа, насыщая его словами, которые вероятно будут в запросах пользователей, что позволяет BM25 находить документ по этим словам.

### Архитектура
Doc2Query использует Seq2Seq (Encoder-Decoder):
1. **Encoder**: принимает текст документа.
2. **Decoder**: генерирует вероятный запрос.
3. Позже использовалась T5 (2019, Google), улучшившая качество генерации.

### Обучение
Модель обучается как Machine Translation:
1. Датасет MS MARCO с парами (Document, Relevant Query).
2. Input: текст документа.
3. Target: реальный запрос.
4. Loss: Cross-Entropy Loss.

### Инференс
Процесс выполняется Offline:
1. Документ подается в модель.
2. Генерируются 10-40 запросов.
3. Запросы добавляются к тексту документа.
4. Расширенный текст индексируется поисковым движком.
5. Поиск с BM25, вероятность мэтчинга увеличивается.

### Результаты
Эксперименты на MS MARCO и TREC CAR:
- MRR@10 увеличился с 18.7% до 21.5% (~15% улучшение).
- Recall@1000 значительно вырос, улучшив качество кандидатов для переранжирования.
- Нулевое влияние на Latency поиска, оверхед перенесен на индексацию.

<img src="img/img.png" width=500>
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример использования Doc2Query для расширения документа с помощью предсказанных запросов.
# Мы будем использовать библиотеку Hugging Face Transformers для загрузки предобученной модели T5,
# которая может быть использована для генерации запросов.

from transformers import T5ForConditionalGeneration, T5Tokenizer

# Загружаем предобученную модель T5 и токенизатор
model_name = "t5-base"
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)

# Пример документа, который мы хотим расширить
document_text = """
Python является высокоуровневым языком программирования, который используется для разработки веб-приложений,
научных вычислений, анализа данных и автоматизации задач. Он известен своей простотой и читаемостью кода.
"""

# Подготавливаем входные данные для модели
input_text = "document: " + document_text
input_ids = tokenizer.encode(input_text, return_tensors="pt")

# Генерируем запросы с помощью модели
# Используем beam search для генерации нескольких вариантов запросов
outputs = model.generate(input_ids, max_length=64, num_beams=5, num_return_sequences=3)

# Декодируем и выводим сгенерированные запросы
generated_queries = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
print("Generated Queries:")
for i, query in enumerate(generated_queries):
    print(f"{i + 1}: {query}")

# Конкатенируем сгенерированные запросы с оригинальным текстом документа
expanded_document = document_text + "\n" + "\n".join(generated_queries)

print("\nExpanded Document:")
print(expanded_document)

# В реальном сценарии расширенный документ будет индексироваться в поисковой системе,
# такой как Elasticsearch, с использованием стандартного BM25.
```

### Комментарии к коду:
1. **Модель и токенизатор**: Мы используем предобученную модель T5, которая была выбрана в более поздних версиях Doc2Query (DocTTTTTquery) для улучшения качества генерации запросов.

2. **Подготовка данных**: Мы добавляем префикс "document: " к тексту документа, чтобы модель понимала, что это входной текст, который нужно преобразовать в запросы.

3. **Генерация запросов**: Используем beam search для генерации нескольких вариантов запросов. Это позволяет получить разнообразные и потенциально релевантные запросы.

4. **Расширение документа**: Сгенерированные запросы добавляются к оригинальному тексту документа. Этот расширенный текст затем может быть проиндексирован в поисковой системе.

5. **Преимущества**: Основное преимущество метода Doc2Query заключается в том, что он улучшает полноту поиска без увеличения времени отклика, так как все вычисления выполняются на этапе индексации.